# MIAAM — per-student sequence preparation

This notebook processes the MIAAM/NeurIPS dataset of primary-school students completing adaptive math exercises and produces per-student interaction sequences for Knowledge Tracing experiments.

It outputs three parquet files into `../data/`:
- `sequences.parquet` — full dataset (~38 k students), one row per student; attempts stored as a chronologically ordered list of structs
- `train_sequences.parquet` — 80 % random split (seed 42)
- `val_sequences.parquet` — 20 % random split (seed 42)

Each attempt struct carries: `exercise_id_int`, `exercise_id`, `activity_id_int`, `activity_id`, `data_correct`, `created_at` (Unix seconds), `data_duration`, `work_mode`, `data_answer`, `session_id`, `module_name`.

## 1) Load MIAAM dataset

Loads the three source files from `../HF_neurips_official_dataset/`:
- `maths_data_filtered.parquet` — 36,837 students, 6,656,038 attempts (students with <5 attempts or adaptive-test-only histories excluded)
- `maths_exercises_table.parquet` — exercise metadata including hierarchy IDs (`objective_id`, `activity_id`, `module_id`) and text content
- `maths_dependencies.json` — activity-level pedagogical dependency graph

In [ ]:
import polars as pl
import json
import pathlib
import os
import numpy as np


DATASET = pathlib.Path("../../MIAAM")
SAVE_FOLDER = pathlib.Path("../data")

interactions = pl.read_parquet(DATASET / "data/maths_data_filtered.parquet")
exercises    = pl.read_parquet(DATASET / "data/maths_exercises_table.parquet")
deps         = json.loads((DATASET / "data/maths_dependencies.json").read_text())

print(interactions.shape, exercises.shape)

(6481693, 14) (7151, 12)


In [2]:
interactions.head(2)

user_id,classroom_id,playlist_or_module_id,exercise_id,created_at,data_correct,work_mode,data_answer,data_duration,source,attempt_index,session_id,module_id,module_name
str,str,str,str,datetime[μs],bool,str,str,f64,str,u32,str,str,str
"""36705bac-0394-4d0a-8365-3be80a…","""58f3c4d4-01dc-4be9-8435-f7f46f…","""63e98e5f-94e3-4630-9704-076882…","""97516718-fd8f-4f90-8ead-56dff6…",2023-12-01 00:00:12.815,true,"""zpdes""","""[1]""",2335.0,"""am""",1,"""am::36705bac-0394-4d0a-8365-3b…","""63e98e5f-94e3-4630-9704-076882…","""Nombres et calcul"""
"""36705bac-0394-4d0a-8365-3be80a…","""58f3c4d4-01dc-4be9-8435-f7f46f…","""63e98e5f-94e3-4630-9704-076882…","""f90ab87b-7054-4de8-b836-300653…",2023-11-24 00:01:50.092,true,"""zpdes""","""[1]""",3508.0,"""am""",1,"""am::36705bac-0394-4d0a-8365-3b…","""63e98e5f-94e3-4630-9704-076882…","""Nombres et calcul"""


In [3]:
exercises.head(2)

exercise_id,gameplay_type,content,module_id,module_name,objective_id,objective_name,objective_pedagogical_intent,activity_id,activity_name,activity_pedagogical_intent,source
str,str,str,str,str,str,str,str,str,str,str,str
"""39676249-728a-4d21-9a58-c31451…","""INTERVAL_COLORING""","""{""instruction"": ""[description …","""63e98e5f-94e3-4630-9704-076882…","""Nombres et calcul""","""5d6e6d4b-b905-41c1-81d6-024fbe…","""Découvrir la multiplication""","""En utilisant la ligne numériqu…","""8e532d14-06dc-45db-a1c9-bdf4c7…","""Toutes tables""","""Décomposer toutes les tables s…","""am"""
"""1c921f06-1005-11ed-861d-0242ac…","""MULTIPLE_CHOICE""","""{""instruction"": ""Clique sur le…","""63e98e5f-94e3-4630-9704-076882…","""Nombres et calcul""","""57c36436-8e30-418d-9656-b888de…","""Comparer des nombres (1-100)""","""Comparer des quantités (chiffr…","""a8e9e045-c72f-4722-bb0a-4173d0…","""Nombres entre 60 et 99 (chiffr…","""Comparer des nombres (représen…","""am"""


### Filter exercises without a screenshot

Some exercises in `maths_exercises_table.parquet` have no corresponding screenshot in `data/screenshots/compressed/`. They are removed from both `exercises` and `interactions` before any further processing, so that all remaining exercises have visual content available for multimodal models.

In [4]:
SCREENSHOTS = DATASET / "data/screenshots/compressed"
screenshot_ids = {
    f.stem
    for source_dir in SCREENSHOTS.iterdir()
    for f in source_dir.iterdir()
    if f.suffix == ".png"
}

missing_screenshots = exercises.filter(~pl.col("exercise_id").is_in(list(screenshot_ids)))
print(f"Exercises without a raw screenshot: {len(missing_screenshots)}")
print(missing_screenshots.select(["exercise_id", "source", "module_name", "activity_name"]))

exercises    = exercises.filter(pl.col("exercise_id").is_in(list(screenshot_ids)))
interactions = interactions.filter(pl.col("exercise_id").is_in(list(screenshot_ids)))
print(f"\nAfter filtering: {interactions.shape[0]} attempts, {interactions['exercise_id'].n_unique()} unique exercises")

Exercises without a raw screenshot: 12
shape: (12, 4)
┌─────────────────────────────────┬────────┬───────────────────┬─────────────────────────────────┐
│ exercise_id                     ┆ source ┆ module_name       ┆ activity_name                   │
│ ---                             ┆ ---    ┆ ---               ┆ ---                             │
│ str                             ┆ str    ┆ str               ┆ str                             │
╞═════════════════════════════════╪════════╪═══════════════════╪═════════════════════════════════╡
│ a84300cf-53ce-497d-aa20-1778f1… ┆ am     ┆ Nombres et calcul ┆ Table de 5                      │
│ fa2d0f28-0e2c-4d85-a1ef-0df785… ┆ am     ┆ Nombres et calcul ┆ Points disposés aléatoirement … │
│ 87fd4c76-c5dc-4cc4-b056-7096c5… ┆ am     ┆ Nombres et calcul ┆ Points disposés aléatoirement … │
│ e5fae336-e143-4070-b7f3-4ce23a… ┆ am     ┆ Nombres et calcul ┆ Table de 7                      │
│ 415808bf-9327-4e3a-a6af-694cf6… ┆ am     ┆ Nombres et

## 2) Build per-student sequences

Enriches the attempt log with integer-encoded IDs and restructures it into per-student sequences suitable as input for Knowledge Tracing models. The result is a Polars DataFrame in parquet format — one row per student, all attempt fields collected into a parallel-list struct column ordered chronologically by `created_at`.

### Join exercise metadata

Join `activity_id` from the exercise table onto the attempt log using `exercise_id` as the key.

In [5]:
interactions = interactions.join(
    exercises.select(["exercise_id", "activity_id"]),
    on="exercise_id",
    how="left",
)

print(interactions.select(["exercise_id", "activity_id"]).head())

shape: (5, 2)
┌─────────────────────────────────┬─────────────────────────────────┐
│ exercise_id                     ┆ activity_id                     │
│ ---                             ┆ ---                             │
│ str                             ┆ str                             │
╞═════════════════════════════════╪═════════════════════════════════╡
│ 97516718-fd8f-4f90-8ead-56dff6… ┆ 07144c8b-67d5-4596-b9e3-d49841… │
│ f90ab87b-7054-4de8-b836-300653… ┆ 0f54e38b-1a89-40aa-ad52-da52ba… │
│ f90ab87b-7054-4de8-b836-300653… ┆ 0f54e38b-1a89-40aa-ad52-da52ba… │
│ 21a30e28-d557-4760-a3dd-3838f4… ┆ 0f54e38b-1a89-40aa-ad52-da52ba… │
│ 21a30e28-d557-4760-a3dd-3838f4… ┆ 0f54e38b-1a89-40aa-ad52-da52ba… │
└─────────────────────────────────┴─────────────────────────────────┘


### Integer-encode IDs

KT models require contiguous integer indices for students, exercises, and skills (for which we use activities). Each UUID is mapped to a 0-based integer by sorting unique values alphabetically and assigning row indices — producing a stable mapping independent of the order of the interaction data.

In [6]:
user_id_map = (
    interactions.select("user_id")
    .unique()
    .sort("user_id")
    .with_row_index("user_id_int")
)

interactions = interactions.join(user_id_map, on="user_id", how="left")

print(f"{interactions['user_id_int'].n_unique()} unique students")
print(interactions.select(["user_id", "user_id_int"]).head())

38520 unique students
shape: (5, 2)
┌─────────────────────────────────┬─────────────┐
│ user_id                         ┆ user_id_int │
│ ---                             ┆ ---         │
│ str                             ┆ u32         │
╞═════════════════════════════════╪═════════════╡
│ 36705bac-0394-4d0a-8365-3be80a… ┆ 8097        │
│ 36705bac-0394-4d0a-8365-3be80a… ┆ 8097        │
│ 36705bac-0394-4d0a-8365-3be80a… ┆ 8097        │
│ 36705bac-0394-4d0a-8365-3be80a… ┆ 8097        │
│ 36705bac-0394-4d0a-8365-3be80a… ┆ 8097        │
└─────────────────────────────────┴─────────────┘


In [7]:
exercise_id_map = (
    interactions.select("exercise_id")
    .unique()
    .sort("exercise_id")
    .with_row_index("exercise_id_int")
)

interactions = interactions.join(exercise_id_map, on="exercise_id", how="left")

print(f"{interactions['exercise_id_int'].n_unique()} unique exercises")
print(interactions.select(["exercise_id", "exercise_id_int"]).head())

6936 unique exercises
shape: (5, 2)
┌─────────────────────────────────┬─────────────────┐
│ exercise_id                     ┆ exercise_id_int │
│ ---                             ┆ ---             │
│ str                             ┆ u32             │
╞═════════════════════════════════╪═════════════════╡
│ 97516718-fd8f-4f90-8ead-56dff6… ┆ 4022            │
│ f90ab87b-7054-4de8-b836-300653… ┆ 6723            │
│ f90ab87b-7054-4de8-b836-300653… ┆ 6723            │
│ 21a30e28-d557-4760-a3dd-3838f4… ┆ 1033            │
│ 21a30e28-d557-4760-a3dd-3838f4… ┆ 1033            │
└─────────────────────────────────┴─────────────────┘


In [8]:
activity_id_map = (
    interactions.select("activity_id")
    .unique()
    .sort("activity_id")
    .with_row_index("activity_id_int")
)

interactions = interactions.join(activity_id_map, on="activity_id", how="left")

print(f"{interactions['activity_id_int'].n_unique()} unique activities")
print(interactions.select(["exercise_id", "activity_id", "activity_id_int"]).head())

351 unique activities
shape: (5, 3)
┌─────────────────────────────────┬─────────────────────────────────┬─────────────────┐
│ exercise_id                     ┆ activity_id                     ┆ activity_id_int │
│ ---                             ┆ ---                             ┆ ---             │
│ str                             ┆ str                             ┆ u32             │
╞═════════════════════════════════╪═════════════════════════════════╪═════════════════╡
│ 97516718-fd8f-4f90-8ead-56dff6… ┆ 07144c8b-67d5-4596-b9e3-d49841… ┆ 6               │
│ f90ab87b-7054-4de8-b836-300653… ┆ 0f54e38b-1a89-40aa-ad52-da52ba… ┆ 22              │
│ f90ab87b-7054-4de8-b836-300653… ┆ 0f54e38b-1a89-40aa-ad52-da52ba… ┆ 22              │
│ 21a30e28-d557-4760-a3dd-3838f4… ┆ 0f54e38b-1a89-40aa-ad52-da52ba… ┆ 22              │
│ 21a30e28-d557-4760-a3dd-3838f4… ┆ 0f54e38b-1a89-40aa-ad52-da52ba… ┆ 22              │
└─────────────────────────────────┴─────────────────────────────────┴───────────────

### Build per-student sequences

Group the enriched attempt log by student and collect each field into a parallel list ordered chronologically by `created_at` (converted to a Unix timestamp in seconds). The result is a dict keyed by `user_id_int`, where each value is a dict of lists — one list per field, all of the same length.

In [9]:
# student_sequences = {
#     row["user_id_int"]: {
#         "exercise_id_int":      [a["exercise_id_int"]      for a in row["attempts"]],
#         "exercise_id":          [a["exercise_id"]      for a in row["attempts"]],  
#         "activity_id_int":      [a["activity_id_int"]      for a in row["attempts"]],
#         "activity_id":          [a["activity_id"]      for a in row["attempts"]],
#         "data_correct":         [int(a["data_correct"])     for a in row["attempts"]],
#         "created_at":           [a["created_at"]           for a in row["attempts"]],
#         "data_duration":        [int(a["data_duration"])    for a in row["attempts"]],
#     }
#     for row in (
#         interactions
#         .select(["user_id_int", "exercise_id_int", "exercise_id", "activity_id_int", "activity_id", "data_correct", "created_at", "data_duration"])
#         .with_columns(pl.col("created_at").dt.epoch(time_unit="s").alias("created_at"))
#         .sort(["user_id_int", "created_at"])
#         .group_by("user_id_int", maintain_order=True)
#         .agg(pl.struct(["exercise_id_int", "exercise_id", "activity_id_int", "activity_id", "data_correct", "created_at", "data_duration"]).alias("attempts"))
#         .iter_rows(named=True)
#     )
# }

# print(f"{len(student_sequences)} students")
# uid = next(iter(student_sequences))
# print(f"Example — student {uid}:")
# print({k: v[:2] for k, v in student_sequences[uid].items()})

In [10]:
seq_df = (
    interactions
    .select([
        "user_id_int", "exercise_id_int", "exercise_id",
        "activity_id_int", "activity_id", "data_correct",
        "created_at", "data_duration",
        "work_mode", "data_answer", "session_id", "module_name",  # extras you want in history
    ])
    .with_columns(pl.col("created_at").dt.epoch(time_unit="s").alias("created_at"))
    .sort(["user_id_int", "created_at"])
    .group_by("user_id_int", maintain_order=True)
    .agg(pl.struct([
        "exercise_id_int", "exercise_id", "activity_id_int", "activity_id",
        "data_correct", "created_at", "data_duration",
        "work_mode", "data_answer", "session_id", "module_name",
    ]).alias("attempts"))
)

In [11]:
seq_df.head(2)

user_id_int,attempts
u32,list[struct[11]]
0,"[{3197,""77a65f22-ef59-4958-8c0d-a48bf52a29b3"",22,""0f54e38b-1a89-40aa-ad52-da52baf76155"",true,1697417014,6280.0,""zpdes"",""[1]"",""am::00008169-3452-462c-9b68-775c3d7274f0::2023-10-16T18:59:10.256000+00:00"",""Nombres et calcul""}, {4699,""b620b175-7758-48b0-b7da-33ba4aecdcb3"",22,""0f54e38b-1a89-40aa-ad52-da52baf76155"",true,1697417022,3825.0,""zpdes"",""[1]"",""am::00008169-3452-462c-9b68-775c3d7274f0::2023-10-16T18:59:10.256000+00:00"",""Nombres et calcul""}, … {1363,""302022f4-94be-4e40-a3ca-b2e0341f4a6b"",217,""9b014385-897b-4474-a1e7-285e0847aa08"",true,1704844996,1973.0,""zpdes"",""[0]"",""am::00008169-3452-462c-9b68-775c3d7274f0::2024-01-10T17:18:17.691000+00:00"",""Nombres et calcul""}]"
1,"[{3706,""8991a32c-eb0e-11ec-8fea-0242ac120002"",278,""cdb2070d-3e38-4dda-8db4-64d32a470ce4"",true,1763424397,19037.0,""adaptive-test"",""{'dropzoneAnswers': [{'id': '1', 'tag': '8', 'state': 'dropzone', 'value': '4', 'tagValue': '4'}, {'id': '3', 'tag': '6', 'state': 'dropzone', 'value': '16', 'tagValue': '16'}, {'id': '5', 'tag': '7', 'state': 'dropzone', 'value': '20 – 1', 'tagValue': '20 – 1'}]}"",""am::0001d9aa-3144-423e-9f13-4d8ad099983a::2025-11-18T09:56:34.484000+00:00"",""Nombres et calcul""}, {2609,""62ebe19b-837b-413d-998f-c2750ce0c677"",217,""9b014385-897b-4474-a1e7-285e0847aa08"",true,1763424464,27040.0,""adaptive-test"",""[1]"",""am::0001d9aa-3144-423e-9f13-4d8ad099983a::2025-11-18T09:56:34.484000+00:00"",""Nombres et calcul""}, … {673,""1a26c14c-11b6-11ed-861d-0242ac120002"",181,""8244f77b-1564-4cf2-951b-573de4106775"",true,1763427215,2350.0,""zpdes"",""[0]"",""am::0001d9aa-3144-423e-9f13-4d8ad099983a::2025-11-18T09:56:34.484000+00:00"",""Nombres et calcul""}]"


In [12]:
seq_df.write_parquet(SAVE_FOLDER / "sequences.parquet")

# --- static exercise metadata, untouched ---
exercises.write_parquet(SAVE_FOLDER / "exercises.parquet")

In [15]:
shuffled = seq_df.sample(fraction=1.0, shuffle=True, seed=42)
n_val = int(len(shuffled) * 0.2)

val_df = shuffled[:n_val]
train_df = shuffled[n_val:]

train_df.write_parquet(SAVE_FOLDER / "train_sequences.parquet")
val_df.write_parquet(SAVE_FOLDER / "val_sequences.parquet")

In [16]:
train_df.shape

(30816, 2)

In [17]:
val_df.shape

(7704, 2)